## MobileNetV2 model
---
![mobnv2](../picture/MobileNetV2.png)
MobileNetV2 is a convolutional neural network architecture that seeks to perform well on mobile devices. It is based on an inverted residual structure where the residual connections are between the bottleneck layers. The intermediate expansion layer uses lightweight depthwise convolutions to filter features as a source of non-linearity. As a whole, the architecture of MobileNetV2 contains the initial fully convolution layer with 32 filters, followed by 19 residual bottleneck layers. [Source](https://paperswithcode.com/method/mobilenetv2#:~:text=MobileNetV2%20is%20a%20convolutional%20neural,are%20between%20the%20bottleneck%20layers.)

In [40]:
# Import library
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image

# Open file
import os
import PIL
from random import seed
# Model CNN (Deep learning network)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense,\
GlobalAveragePooling2D, Dropout, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [41]:
# Create function plot loss function and accuracy score graph
def plot_graph(model_values):
    '''
    Input : Model_values of keras.callbacks.History
    Return : Graph of Loss function and accuracy score between training dataset and vaildation dataset
    '''
    # Subplots
    fig, ax = plt.subplots(1, 2, figsize=(14,5))

    # Plot loss
    plt.subplot(1, 2, 1)
    plt.plot(model_values.history['loss'], label='Training Loss');
    plt.plot(model_values.history['val_loss'], label='Testing Loss');
    plt.legend(fontsize=12, loc='upper right')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss');

    # Plot MSE
    plt.subplot(1, 2, 2)

    plt.plot(model_values.history['accuracy'], label='Training Accuracy')
    plt.plot(model_values.history['val_accuracy'], label='Validation Accuracy')

    plt.legend(fontsize=12, loc='lower right')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy');

### 00 First, check GPU.
---

In [42]:
# https://www.tensorflow.org/guide/keras/sequential_model
# Due to we use Keras Sequential API,
# We want to check GPU first before training our model for
# impore efficiency and reduce time.
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 6933259299507502885
xla_global_id: -1
]


### 01 Open dataset
---

In [43]:
import os
import shutil
import random

# Define dataset path
base_dir = r"D:\DataSet"

# Check if base dataset directory exists
if not os.path.exists(base_dir):
    raise FileNotFoundError(f"Error: The dataset directory '{base_dir}' does not exist.")

# Define train, valid, and test directories
train_dir = os.path.join(base_dir, "train")
valid_dir = os.path.join(base_dir, "valid")
test_dir = os.path.join(base_dir, "test")

# Create output directories
for folder in [train_dir, valid_dir, test_dir]:
    os.makedirs(folder, exist_ok=True)

# List of class folders (make sure these exist in base_dir)
class_names = ["cocci", "healthy", "ncd", "salmo"]

# Split ratios
train_ratio = 0.7
valid_ratio = 0.2
test_ratio = 0.1

# Process each class
for class_name in class_names:
    class_path = os.path.join(base_dir, class_name)

    # Check if class folder exists
    if not os.path.exists(class_path):
        print(f"⚠️ Class folder '{class_name}' does not exist. Skipping.")
        continue

    # List image files
    images = [
        f for f in os.listdir(class_path)
        if os.path.isfile(os.path.join(class_path, f)) and f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ]

    if not images:
        print(f"⚠️ No image files found in '{class_name}'. Skipping.")
        continue

    # Shuffle for randomness
    random.shuffle(images)

    # Compute split sizes
    total = len(images)
    train_size = int(total * train_ratio)
    valid_size = int(total * valid_ratio)
    test_size = total - train_size - valid_size

    # Split image lists
    train_images = images[:train_size]
    valid_images = images[train_size:train_size + valid_size]
    test_images = images[train_size + valid_size:]

    # Create class subfolders in train, valid, test
    for folder in [train_dir, valid_dir, test_dir]:
        os.makedirs(os.path.join(folder, class_name), exist_ok=True)

    # Copy files to respective folders
    def copy_files(image_list, target_dir):
        for img in image_list:
            src = os.path.join(class_path, img)
            dst = os.path.join(target_dir, class_name, img)
            try:
                shutil.copy2(src, dst)
            except Exception as e:
                print(f"❌ Error copying '{img}': {e}")

    # Copy all images
    copy_files(train_images, train_dir)
    copy_files(valid_images, valid_dir)
    copy_files(test_images, test_dir)

    print(f"✅ Class '{class_name}': {len(train_images)} train, {len(valid_images)} valid, {len(test_images)} test")

print("\n🎉 Data restructuring complete!")


✅ Class 'cocci': 1472 train, 420 valid, 211 test
✅ Class 'healthy': 1439 train, 411 valid, 207 test
✅ Class 'ncd': 369 train, 105 valid, 54 test
✅ Class 'salmo': 1593 train, 455 valid, 228 test

🎉 Data restructuring complete!


### 02 Preprocessing image dataset
---

In [ ]:
import os
import shutil
import random
from PIL import Image, ImageOps
from concurrent.futures import ThreadPoolExecutor

base_dir = r"D:\DataSet"
train_dir, valid_dir, test_dir = [os.path.join(base_dir, x) for x in ["train", "valid", "test"]]
for folder in [train_dir, valid_dir, test_dir]:
    os.makedirs(folder, exist_ok=True)

class_names = ["cocci", "healthy", "ncd", "salmo"]
train_ratio, valid_ratio, test_ratio = 0.7, 0.2, 0.1

def augment_image(img):
    funcs = [
        lambda x: x.rotate(90), lambda x: x.rotate(180), lambda x: x.rotate(270),
        lambda x: ImageOps.mirror(x), lambda x: ImageOps.flip(x)
    ]
    return random.choice(funcs)(img)

def process_and_save(img_obj, dst_path, augment=False):
    try:
        if augment:
            img_obj = augment_image(img_obj)
        img_obj.save(dst_path)
    except Exception as e:
        print(f"❌ Error saving '{dst_path}': {e}")

def process_split(image_list, src_class_path, dst_dir, class_name, target_count):
    class_dst = os.path.join(dst_dir, class_name)
    os.makedirs(class_dst, exist_ok=True)
    base_count = len(image_list)
    
    # Preload images once
    preloaded_images = []
    for fname in image_list:
        try:
            img_path = os.path.join(src_class_path, fname)
            img = Image.open(img_path).convert("RGB")
            preloaded_images.append((fname, img))
        except Exception as e:
            print(f"❌ Error opening '{fname}': {e}")
    
    tasks = []
    with ThreadPoolExecutor() as executor:
        for i in range(target_count):
            idx = i % base_count
            fname, img = preloaded_images[idx]
            aug = i >= base_count
            new_name = f"aug_{i}_{fname}" if aug else fname
            dst_path = os.path.join(class_dst, new_name)
            tasks.append(executor.submit(process_and_save, img.copy(), dst_path, aug))

def split_data(images):
    random.shuffle(images)
    total = len(images)
    return (
        images[:int(total * train_ratio)],
        images[int(total * train_ratio):int(total * (train_ratio + valid_ratio))],
        images[int(total * (train_ratio + valid_ratio)):]
    )

split_sizes = {'train': {}, 'valid': {}, 'test': {}}
data_splits = {}

for class_name in class_names:
    src_class = os.path.join(base_dir, class_name)
    if not os.path.exists(src_class):
        print(f"⚠️ Missing folder '{class_name}'. Skipping.")
        continue

    images = [f for f in os.listdir(src_class)
              if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if not images:
        print(f"⚠️ No images in '{class_name}'. Skipping.")
        continue

    train_imgs, valid_imgs, test_imgs = split_data(images)
    data_splits[class_name] = {'train': train_imgs, 'valid': valid_imgs, 'test': test_imgs}
    split_sizes['train'][class_name] = len(train_imgs)
    split_sizes['valid'][class_name] = len(valid_imgs)
    split_sizes['test'][class_name] = len(test_imgs)

max_train = max(split_sizes['train'].values())
max_valid = max(split_sizes['valid'].values())
max_test = max(split_sizes['test'].values())

for class_name in class_names:
    class_path = os.path.join(base_dir, class_name)
    if class_name not in data_splits:
        continue

    process_split(data_splits[class_name]['train'], class_path, train_dir, class_name, max_train)
    process_split(data_splits[class_name]['valid'], class_path, valid_dir, class_name, max_valid)
    process_split(data_splits[class_name]['test'], class_path, test_dir, class_name, max_test)
    
    print(f"✅ Balanced '{class_name}': {max_train} train, {max_valid} valid, {max_test} test")

print("\n🎉 Optimized data balancing complete!")


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Parameters
batch_size = 50
img_height = 128
img_width = 128
target_size = (img_height, img_width)
seed = 42
# Dataset directories (must contain subfolders: cocci, healthy, ncd, salmo)
train_dir = r"D:\DataSet\train"
valid_dir = r"D:\DataSet\valid"
test_dir  = r"D:\DataSet\test"
# ✅ Check that directories exist
for path in [train_dir, valid_dir, test_dir]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Directory not found: {path}")
# ✅ Normalize pixel values
data_gen_train = ImageDataGenerator(rescale=1./255)
data_gen_valid = ImageDataGenerator(rescale=1./255)
data_gen_test  = ImageDataGenerator(rescale=1./255)
# ✅ Load validation data (no shuffling!)
valid_generator = data_gen_valid.flow_from_directory(
    directory=valid_dir,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False,
    seed=seed
)
# ✅ Load training data
train_generator = data_gen_train.flow_from_directory(
    directory=train_dir,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True,
    seed=seed
)
# ✅ Load test data (no shuffling!)
test_generator = data_gen_test.flow_from_directory(
    directory=test_dir,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False,
    seed=seed
)
# ✅ Check class indices (important for interpreting predictions later)
print("Class mapping:", train_generator.class_indices)


---

In [ ]:
import os

def count_classes_in_folder(folder_path):
    class_counts = {}
    if not os.path.exists(folder_path):
        print(f"❌ Error: Folder does not exist — {folder_path}")
        return class_counts

    for class_name in os.listdir(folder_path):
        class_dir = os.path.join(folder_path, class_name)
        if os.path.isdir(class_dir):
            # Count only files (ignore subfolders)
            count = len([
                f for f in os.listdir(class_dir)
                if os.path.isfile(os.path.join(class_dir, f))
            ])
            class_counts[class_name] = count
    return class_counts

# Use your actual dataset paths (case sensitive)
base_path = r"D:\DataSet"
test_counts  = count_classes_in_folder(os.path.join(base_path, "test"))
train_counts = count_classes_in_folder(os.path.join(base_path, "train"))
valid_counts = count_classes_in_folder(os.path.join(base_path, "valid"))

print("Class Distribution")
print("Train:", train_counts)
print("Valid:", valid_counts)
print("Test: ", test_counts)




### 03 MobileNetV2 model training
---

#### Transfer learning

In [ ]:
# https://keras.io/api/applications/mobilenet/#mobilenetv2-function
# import MobileNetV2 model form keras API
# set input size of image of trianing is 128x128 (smallest size of MobileNetV2)
# due to we want to use transfer learning process
# we must add `include_top=False` because we wan to add our input data
# we decide default weigh for mode
mobv2_model = tf.keras.applications.MobileNetV2(input_shape=(128,128,3),
                                                include_top=False, # Transfer learning
                                                weights="imagenet")

In [ ]:
# model summary
# Total params: 2,257,984
# Trainable params: 2,223,872
# Non-trainable params: 34,112
mobv2_model.summary()

In [ ]:
# fix weights and bias
# train specifically custom head
mobv2_model.trainable=False

#### Add custom head and output layers

In [ ]:
from tensorflow.keras import layers, regularizers

# Flatten the output from MobileNetV2
x = tf.keras.layers.GlobalAveragePooling2D()(mobv2_model.output)

# Add a dense layer with regularization
x = layers.Dense(
    128,
    activation='relu',
    kernel_regularizer=regularizers.l2(0.001)  # L2 regularization to prevent large weights
)(x)

# Add batch normalization to stabilize and regularize
x = layers.BatchNormalization()(x)

# Add dropout to randomly deactivate neurons during training
x = layers.Dropout(0.3)(x)

# Final classification layer
prediction_layer = layers.Dense(units=4, activation="softmax")(x)


In [ ]:
# Add Input layer and output layer
model = tf.keras.models.Model(inputs=mobv2_model.input,
                                    outputs=prediction_layer)

In [ ]:
# Total params: 2,263,108
# Trainable params: 5,124 # add input layers and  output layers
# Non-trainable params: 2,257,984 --> fix layers
model.summary()

In [ ]:
# Compile the model
model.compile(loss="categorical_crossentropy",
              optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              metrics=['accuracy'])

In [ ]:
import os
import tensorflow as tf

# Save checkpoints during training
checkpoint_path = "D:/model/poultrynet1_cp/cp.weights.h5"  # ← FIXED extension
checkpoint_dir = os.path.dirname(checkpoint_path)

# Create a callback that saves the model's weights
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    mode="max",
    verbose=1,
    monitor="val_accuracy"
)


In [ ]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # Handle broken images

# Data generators (normalize pixel values)
train_datagen = ImageDataGenerator(rescale=1./255)
valid_datagen = ImageDataGenerator(rescale=1./255)

# Load training and validation data
train_dataset = train_datagen.flow_from_directory(
    'D:/Dataset/train',
    target_size=(224, 224),        # 👈 Must match model input size
    batch_size=32,
    class_mode='categorical'
)

valid_dataset = valid_datagen.flow_from_directory(
    'D:/Dataset/valid',
    target_size=(224, 224),        # 👈 Must match model input size
    batch_size=32,
    class_mode='categorical'
)

# Define CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(train_dataset.num_classes, activation='softmax')  # Dynamic class count
])

# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Define checkpoint callback
checkpoint_path = "best_model.h5"
cp_callback = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=False,  # Set to True if you only want weights
    mode='max',
    verbose=1
)

# Train the model
history = model.fit(
    train_dataset,
    epochs=25,
    validation_data=valid_dataset,
    callbacks=[cp_callback]
)


In [ ]:
# plot graph
plot_graph(history)


# Overfitting between training and validation
# Final accuracy after training 25 epochs was score in training, 0.98%
# and vaildaion 0.90%
# Loss function after training 25 epochs was score in training, 0.06%
# and vaildaion 0.31%
# Good perfomance than baseline model with better result.

In [ ]:
# convert the history.history dict to a pandas DataFrame:
hist_df = pd.DataFrame(history.history)
# save history to csv:
hist_csv_file = 'D:/model/poultrynet/history_poultrynet1_tf.csv'
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)

In [ ]:
# save model
model.save("D:/model/poultrynet/poultrynet1_model.h5")

In [ ]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the .tflite file
with open('D:/model/poultrynet/poultrynet1_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("Model converted to TFLite and saved successfully.")

In [ ]:
import numpy as np
import tensorflow as tf
import os
import warnings
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.metrics import classification_report

# --- OPTIONAL: Suppress warnings ---
warnings.filterwarnings("ignore", category=UserWarning)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# --- PARAMETERS ---
tflite_model_path = 'D:/model/poultrynet/poultrynet1_model.tflite'
image_dir = 'D:/Dataset/test'
img_size = (224, 224)  # Fixed input size
class_names = ['cocci', 'healthy', 'ncd', 'salmo']

# --- LOAD TFLITE MODEL ---
interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# --- PREDICT FUNCTION ---
def predict_tflite(image_path):
    img = load_img(image_path, target_size=img_size)
    img_array = img_to_array(img) / 255.0
    input_data = np.expand_dims(img_array, axis=0).astype(np.float32)

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    return np.argmax(output_data)

# --- LOOP THROUGH TEST IMAGES ---
y_true = []
y_pred = []

for class_index, class_name in enumerate(class_names):
    class_folder = os.path.join(image_dir, class_name)
    for fname in os.listdir(class_folder):
        img_path = os.path.join(class_folder, fname)
        pred_label = predict_tflite(img_path)

        y_pred.append(pred_label)
        y_true.append(class_index)

# --- METRICS ---
report = classification_report(y_true, y_pred, target_names=class_names, digits=2)
print(report)


In [ ]:
import numpy as np
import tensorflow as tf
import os
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

tflite_model_path = 'D:/model/poultrynet/poultrynet1_model.tflite'
image_dir = 'D:/Dataset/test'
class_names = ['cocci', 'healthy', 'ncd', 'salmo']
img_size = (224, 224)  # ✅ Updated to match model's expected input

interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

def predict_tflite(image_path):
    img = load_img(image_path, target_size=img_size)
    img_array = img_to_array(img) / 255.0
    input_data = np.expand_dims(img_array, axis=0).astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    return np.argmax(output_data)

y_true = []
y_pred = []

for true_idx, class_name in enumerate(class_names):
    folder = os.path.join(image_dir, class_name)
    for fname in os.listdir(folder):
        img_path = os.path.join(folder, fname)
        pred_idx = predict_tflite(img_path)
        y_true.append(true_idx)
        y_pred.append(pred_idx)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - TFLite Inference')
plt.tight_layout()
plt.show()


In [ ]:
def predict_with_ood(image_path, threshold=0.80):
    img = load_img(image_path, target_size=img_size)
    img_array = img_to_array(img) / 255.0
    input_data = np.expand_dims(img_array, axis=0).astype(np.float32)

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    probs = output_data[0]
    max_prob = np.max(probs)
    pred_idx = np.argmax(probs)

    if max_prob < threshold:
        return "OoD", max_prob  # Reject as OoD
    else:
        return class_names[pred_idx], max_prob  # Acceptable prediction

In [ ]:
img_path = r"C:\Users\Salman\Downloads\women.jpg"
label, confidence = predict_with_ood(img_path)

if label == "OoD":
    print(f"Rejected input as OoD with confidence {confidence:.2f}")
else:
    print(f"Predicted: {label} (confidence: {confidence:.2f})")


---

Fine tuning

In [ ]:
import tensorflow as tf

# Set image dimensions
img_height = 128
img_width = 128

# Load MobileNetV2 with pretrained weights (excluding the top classification layers)
mobv2_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights="imagenet"
)

print("Number of layers in the base model:", len(mobv2_model.layers))

# Freeze early layers
fine_tune_at = 100
for layer in mobv2_model.layers[:fine_tune_at]:
    layer.trainable = False

# Print last few layers to verify
for i, layer in enumerate(mobv2_model.layers):
    if i >= fine_tune_at - 5:
        print(i, layer.name, "Trainable:", layer.trainable)

# Add new top layers with regularization
x = tf.keras.layers.GlobalAveragePooling2D()(mobv2_model.output)
x = tf.keras.layers.Dropout(0.5)(x)  # Increased dropout to 0.5
x = tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001))(x)  # Optional extra dense layer with L2 regularization
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(units=4, activation='softmax')(x)

# Build model
finetune_model = tf.keras.Model(inputs=mobv2_model.input, outputs=output)

# Compile
finetune_model.compile(
    loss="categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # Good LR for fine-tuning
    metrics=["accuracy"]
)

In [ ]:
# Save checkpoints during training
# follow value of vaildation scorce
checkpoint = tf.keras.callbacks.ModelCheckpoint('D:/model/poultrynet/poultrynet1_model_ft.h5',
                             monitor= 'val_accuracy',
                             mode= 'max',
                             save_best_only = True,
                             verbose= 1)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, GlobalAveragePooling2D
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Set up ImageDataGenerators for training and validation datasets
train_datagen = ImageDataGenerator(rescale=1./255)
valid_datagen = ImageDataGenerator(rescale=1./255)

# Load images from the dataset (make sure the path is correct)
train_dataset = train_datagen.flow_from_directory(
    'D:/Dataset/train',  # Set the correct path to your training data
    target_size=(128, 128), 
    batch_size=32,
    class_mode='categorical'
)

valid_dataset = valid_datagen.flow_from_directory(
    'D:/Dataset/valid/',  # Set the correct path to your validation data
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
)

# Check dataset shapes
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(valid_dataset)}")
print(f"Train images batch shape: {train_dataset[0][0].shape}")
print(f"Valid images batch shape: {valid_dataset[0][0].shape}")

# Define your model (e.g., fine-tuning a pre-trained model)
base_model = tf.keras.applications.MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
base_model.trainable = False  # Freeze base model layers

# Add custom classification layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
x = Dense(4, activation='softmax')(x)  # 4 classes in the output

finetune_model = Model(inputs=base_model.input, outputs=x)

# Compile the model
finetune_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Set up ModelCheckpoint and EarlyStopping callbacks
checkpoint_path = 'D:/model/checkpoint.h5'
cp_callback = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_accuracy',  # or use 'val_loss' depending on your metric
    save_best_only=True,
    verbose=1
)

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history_ft = finetune_model.fit(
    train_dataset,
    epochs=25,
    validation_data=valid_dataset,
    callbacks=[cp_callback, early_stopping],
    verbose=1  # Print the training progress
)

# Check training results
print("Training complete!")


In [ ]:
finetune_model.summary()

In [ ]:
# plot graph
plot_graph(history_ft)

# Slightly Overfitting between training and validation
# Final accuracy after fine tuning 25 epochs is up score in training 1.00% (at 23 epoch)
# and vaildaion 0.93% (better than transfer learning)
# Loss function after training 25 epochs is down score in training 0.01%  (at 23 epoch)
# and vaildaion 0.78% (worse than transfer learning)
# Higher perfomance than baseline model

In [ ]:
# convert the history.history dict to a pandas DataFrame:
hist_df = pd.DataFrame(history_ft.history)
hist_df.head()

# save to csv:
hist_csv_file = 'D/model/poultrynet/history_poultrynet1_model_ft.csv'
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)

In [ ]:
import os
import pandas as pd

# Convert the history.history dict to a pandas DataFrame
hist_df = pd.DataFrame(history_ft.history)
hist_df.head()

# Create directory if it doesn't exist
output_dir = 'D/model/poultrynet'
os.makedirs(output_dir, exist_ok=True)

# Save to CSV
hist_csv_file = os.path.join(output_dir, 'history_poultrynet1_model_ft.csv')
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)


In [ ]:
# save model after fine tuning
fineture_model.save("D/model/poultrynet/poultrynet_model_ft.h5")

In [ ]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the .tflite file
with open('D:/model/poultrynet/poultrynet1_model_ft.tflite', 'wb') as f:
    f.write(tflite_model)

print("Model converted to TFLite and saved successfully.")

In [ ]:
import numpy as np
import tensorflow as tf
import os
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.metrics import classification_report

# --- PARAMETERS ---
tflite_model_path = 'D:/model/poultrynet/poultrynet1_model_ft.tflite'
image_dir = 'D:/Dataset/test'  # or valid dir
img_size = (128, 128)
class_names = ['cocci', 'healthy', 'ncd', 'salmo']  # Must match training

# --- LOAD TFLITE MODEL ---
interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# --- PREDICT FUNCTION ---
def predict_tflite(image_path):
    img = load_img(image_path, target_size=img_size)
    img_array = img_to_array(img) / 255.0
    input_data = np.expand_dims(img_array, axis=0).astype(np.float32)

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    return np.argmax(output_data)

# --- LOOP THROUGH TEST IMAGES ---
y_true = []
y_pred = []

for class_index, class_name in enumerate(class_names):
    class_folder = os.path.join(image_dir, class_name)
    for fname in os.listdir(class_folder):
        img_path = os.path.join(class_folder, fname)
        pred_label = predict_tflite(img_path)

        y_pred.append(pred_label)
        y_true.append(class_index)

# --- METRICS ---
report = classification_report(y_true, y_pred, target_names=class_names, digits=2)
print(report)


In [ ]:
import numpy as np
import tensorflow as tf
import os
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

tflite_model_path = 'D:/model/poultrynet/poultrynet1_model_ft.tflite'
image_dir = 'D:/Dataset/test'
class_names = ['cocci', 'healthy', 'ncd', 'salmo']
img_size = (128, 128)

interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

def predict_tflite(image_path):
    img = load_img(image_path, target_size=img_size)
    img_array = img_to_array(img) / 255.0
    input_data = np.expand_dims(img_array, axis=0).astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    return np.argmax(output_data)

y_true = []
y_pred = []

for true_idx, class_name in enumerate(class_names):
    folder = os.path.join(image_dir, class_name)
    for fname in os.listdir(folder):
        img_path = os.path.join(folder, fname)
        pred_idx = predict_tflite(img_path)
        y_true.append(true_idx)
        y_pred.append(pred_idx)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - TFLite Inference')
plt.tight_layout()
plt.show()


In [ ]:
def predict_with_ood(image_path, threshold=0.80):
    img = load_img(image_path, target_size=img_size)
    img_array = img_to_array(img) / 255.0
    input_data = np.expand_dims(img_array, axis=0).astype(np.float32)

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    probs = output_data[0]
    max_prob = np.max(probs)
    pred_idx = np.argmax(probs)

    if max_prob < threshold:
        return "OoD", max_prob  # Reject as OoD
    else:
        return class_names[pred_idx], max_prob  # Acceptable prediction


In [ ]:
img_path = r"C:\Users\Salman\Downloads\women.jpg"
label, confidence = predict_with_ood(img_path)

if label == "OoD":
    print(f"Rejected input as OoD with confidence {confidence:.2f}")
else:
    print(f"Predicted: {label} (confidence: {confidence:.2f})")
